# Restore Grid World


In [8]:
import torch 
import json 
from gridworld import GridWorld

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device) # cuda

with open("gridworld.json", "r") as f:
    json_object = json.load(f)
    env = GridWorld(**json_object, device=device)

cuda


In [13]:
print(env.valid_states)
print(env.grid_size)

tensor([[48, 49],
        [47, 49],
        [47, 48],
        ...,
        [43, 49],
        [44, 49],
        [45, 49]], device='cuda:0')
50


## 1.0 Visualize Grid World

In [27]:
from IPython.display import HTML, display 

valid_states_all = torch.cat((env.valid_states, torch.tensor([env.grid_size-1,env.grid_size-1]).to(device).unsqueeze(dim=0)))

valid_states_all = valid_states_all[:,0]*env.grid_size + valid_states_all[:,1]

# print(valid_states_all)
# print(valid_states_all[:,0])
# print(valid_states_all[:,1])

html_text = "<table>" 
for row in range(env.grid_size):
    html_text += "<tr>"
    for col in range(env.grid_size):
        if row*env.grid_size + col in valid_states_all:
            html_text += "<td bgcolor=\"pearl\">"
        else:
            html_text += "<td>"
        html_text += str(env.reward_map[row*env.grid_size+col].tolist())
        html_text += "</td>"
    html_text += "</tr>"
html_text += "</table>"

display(HTML(html_text))

0,-1,0,0,0,-1,0,0,0,0,0,0,-1,0,-1,-1,-1,-1,0,-1,0,-1,0,0,0,-1,0,0,0,-1,-1,0,-1,0,0,0,0,-1,-1,0,0,0,0,0,0,0,0,0,0,0
-1,0,0,0,0,-1,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,-1,-1,0,-1,0,0,0,0,0,-1,0,-1,0,-1,0,-1,0,-1,-1,0,0,0,-1,-1,0,0,-1,-1
0,0,-1,-1,0,-1,0,-1,0,-1,0,0,0,0,0,0,0,-1,0,0,-1,0,0,-1,0,0,0,0,0,0,0,-1,0,0,-1,0,0,-1,-1,0,0,-1,0,0,0,0,-1,0,0,0
0,0,-1,0,-1,-1,0,0,-1,0,-1,0,0,0,0,-1,0,-1,0,-1,0,-1,0,0,-1,-1,0,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,0,-1
-1,-1,-1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-1,-1,0,-1,0,0,0,0,0,0,0,0,0,0,0,0,-1,0,-1,0,0,0,-1,0,0,-1,0,-1,-1,0,0
0,-1,-1,0,0,-1,0,-1,0,0,-1,0,0,-1,0,0,-1,-1,0,0,-1,0,0,0,-1,0,0,0,0,-1,-1,0,0,-1,0,-1,-1,-1,-1,0,0,-1,0,-1,0,0,0,0,0,-1
-1,-1,0,0,0,0,0,0,0,0,0,0,-1,0,-1,0,0,0,0,0,-1,-1,0,0,-1,-1,0,0,-1,-1,0,-1,0,0,0,0,0,0,0,-1,-1,0,0,0,-1,0,-1,0,-1,-1
0,0,0,0,-1,0,0,0,-1,0,0,0,0,-1,0,0,-1,-1,0,0,-1,0,0,-1,0,0,-1,0,0,0,-1,-1,0,0,-1,0,-1,0,-1,0,0,-1,0,-1,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,-1,-1,0,-1,0,0,-1,0,0,-1,-1,0,0,0,0,0,0,-1,0,0,0,-1,-1,-1,-1,-1,-1,0,-1,-1,0,-1,0,0,0,-1,0,0,0,0
0,0,0,-1,0,-1,0,0,-1,-1,0,0,-1,0,0,0,0,0,0,0,0,0,-1,-1,-1,0,-1,0,0,0,0,0,0,0,0,0,0,-1,0,0,0,0,0,0,-1,-1,0,-1,-1,-1
0,0,0,0,0,-1,0,0,0,0,-1,0,0,-1,0,0,0,-1,0,0,0,-1,0,0,-1,0,0,0,-1,-1,0,-1,-1,0,0,0,0,0,-1,-1,-1,-1,-1,-1,0,0,-1,-1,-1,0


# Define Policy

Build a policy $\pi_{\theta}$

In [91]:
import torch.nn as nn 
from torch.nn import functional as F 

STATE_SIZE = env.grid_size*env.grid_size # total 2500 square
ACTION_SIZE = env.action_size  # 4 options to choose 

#
# Define model
# 

class PolicyNet(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.hidden = nn.Linear(STATE_SIZE, hidden_dim) # 2500 x 64
        self.classify = nn.Linear(hidden_dim, ACTION_SIZE) # 64 x 4 
    
    def forward(self, s): 
        outs = self.hidden(s)
        outs = F.relu(outs)
        logits = self.classify(outs)
        return logits 

#
# Generate model
# 

policy_func = PolicyNet().to(device)

In [92]:
print(policy_func)

PolicyNet(
  (hidden): Linear(in_features=2500, out_features=64, bias=True)
  (classify): Linear(in_features=64, out_features=4, bias=True)
)


# Run before train

In [93]:
def pick_sample_and_logits(policy, s):
    """
    Stochastically pick up action and logits with policy model.

    Parameters
    ----------
    policy: torch.nn.Module
            Policy network to use
    s: torch.tensor((..., STATE_SIZE), dtype=int)
            The feature (one-hot) of state
            The above "..." can have arbitrary share with 0 or 1 dimension.
    
    Returns
    -------
    action: torch.tensor((...), dtype=int)
            The picked-up actions
            If input shape is (*, STATE_SIZE), this shape becomes (*).
    logits: torch.tensor((..., ACTION_SIZE), dtype=float)
            Logits of categorical distribution (used to optimize model).
            If input shape is (*, STATE_SIZE), this shape becomes (*, ACTION_SIZE).
    """
    # get logits from state
    # --> size: (*, ACTION_SIZE)
    logits = policy(s.float())

    # from logits to probabilities
    # --> size: (* ACTION_SIZE)
    probs = F.softmax(logits, dim=-1)

    # pick up action's sample
    # --> size:(*, 1)
    a = torch.multinomial(probs, num_samples=1)
    
    # --> size: (*)
    a = a.squeeze()

    return a, logits

In [104]:
s = env.reset(1)
print(f"get Raw state: {s[0]}")

s_onehot = F.one_hot(s, num_classes=STATE_SIZE)
print(f"shape of one hot: {s_onehot.shape}")
print(f"one hot: {s_onehot}")

device = "cuda"
policy = PolicyNet().to(device)

logits = policy(s_onehot.float())
probs = F.softmax(logits, dim=-1)
print(f"logits shape: {logits.shape}")
print(f"logits are calculated: {logits}")
print(f"probss are calculated: {probs}")

action = torch.multinomial(probs, num_samples=1)
action = action.squeeze()
print(f"Action has been selected: {action}")


get Raw state: 2466
shape of one hot: torch.Size([1, 2500])
one hot: tensor([[0, 0, 0,  ..., 0, 0, 0]], device='cuda:0')
logits shape: torch.Size([1, 4])
logits are calculated: tensor([[ 0.0311, -0.1147,  0.1068, -0.0046]], device='cuda:0',
       grad_fn=<AddmmBackward0>)
probss are calculated: tensor([[0.2559, 0.2212, 0.2760, 0.2469]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)
Action has been selected: 2


In [96]:
def evaluate(policy, batch_size):
    total_reward = torch.tensor(0.0).to(device) # init reward = 0.0
    s = env.reset(batch_size)
    print(f"state: {s.shape}")
    i = 0
    while True:
        i+=1
        s_onehot = F.one_hot(s, num_classes=STATE_SIZE)
        print(f"Checked Tile: {s_onehot.shape}")
        with torch.no_grad():
            a, _ = pick_sample_and_logits(policy, s_onehot)
            print(f"Chosen action: {a[0]}")
        s, r, term, trunc = env.step(a, s)
        total_reward += torch.sum(r)
        done = torch.logical_or(term, trunc)
        work_indices = (done==False).nonzero().squeeze(dim=-1)
        if not (len(work_indices) > 0):
            break
        s = s[work_indices]
    print(f"iteration: {i}")
    return total_reward.item() / batch_size 

avg_reward = evaluate(policy_func, 300)
print(f"Estimate Reward: {avg_reward}")

state: torch.Size([300])
Checked Tile: torch.Size([300, 2500])
Chosen action: 1
Checked Tile: torch.Size([300, 2500])
Chosen action: 2
Checked Tile: torch.Size([300, 2500])
Chosen action: 3
Checked Tile: torch.Size([300, 2500])
Chosen action: 2
Checked Tile: torch.Size([300, 2500])
Chosen action: 3
Checked Tile: torch.Size([300, 2500])
Chosen action: 1
Checked Tile: torch.Size([300, 2500])
Chosen action: 1
Checked Tile: torch.Size([300, 2500])
Chosen action: 2
Checked Tile: torch.Size([300, 2500])
Chosen action: 2
Checked Tile: torch.Size([300, 2500])
Chosen action: 1
Checked Tile: torch.Size([300, 2500])
Chosen action: 3
Checked Tile: torch.Size([300, 2500])
Chosen action: 0
Checked Tile: torch.Size([300, 2500])
Chosen action: 0
Checked Tile: torch.Size([300, 2500])
Chosen action: 0
Checked Tile: torch.Size([300, 2500])
Chosen action: 0
Checked Tile: torch.Size([300, 2500])
Chosen action: 2
Checked Tile: torch.Size([300, 2500])
Chosen action: 1
Checked Tile: torch.Size([300, 2500])
Ch